# 상품(products) CRUD

- 조회는 누구나, 등록 / 수정 / 삭제는 **관리자만** (RLS)
- 먼저 `sql/05_make_admin.sql`로 admin01을 관리자로 지정해야 함

In [1]:
from shop_client import new_client, login, TEST_PASSWORD

In [2]:
admin = login("admin01@test.com")
user1 = login("user01@test.com")
guest = new_client()  # 로그인 안 한 사용자

admin_id = admin.auth.get_user().user.id
print("admin01 role:", admin.table("members").select("role").eq("id", admin_id).single().execute().data)

admin01 role: {'role': 'admin'}


## Create: 상품 등록 (관리자)

In [3]:
new_products = [
    {"name": "무선 마우스", "price": 25000, "stock": 10},
    {"name": "기계식 키보드", "price": 89000, "stock": 5},
    {"name": "27인치 모니터", "price": 250000, "stock": 3},
    {"name": "마우스 패드", "price": 9000, "stock": 20},
]
res = admin.table("products").insert(new_products).execute()
for p in res.data:
    print(p["id"], p["name"], p["price"], p["stock"])

1 무선 마우스 25000 10
2 기계식 키보드 89000 5
3 27인치 모니터 250000 3
4 마우스 패드 9000 20


## Read: 조회 (로그인 안 해도 가능)

In [4]:
res = guest.table("products").select("id, name, price, stock").order("id").execute()
print("전체 상품:")
for p in res.data:
    print(" ", p)

res = guest.table("products").select("name, price").lte("price", 30000).execute()
print("3만원 이하:", res.data)

전체 상품:
  {'id': 1, 'name': '무선 마우스', 'price': 25000, 'stock': 10}
  {'id': 2, 'name': '기계식 키보드', 'price': 89000, 'stock': 5}
  {'id': 3, 'name': '27인치 모니터', 'price': 250000, 'stock': 3}
  {'id': 4, 'name': '마우스 패드', 'price': 9000, 'stock': 20}
3만원 이하: [{'name': '무선 마우스', 'price': 25000}, {'name': '마우스 패드', 'price': 9000}]


## Update: 가격, 재고 수정 (관리자)

In [5]:
res = admin.table("products").update({"price": 22000, "stock": 15}).eq("name", "무선 마우스").execute()
print("수정 결과:", res.data)

수정 결과: [{'id': 1, 'name': '무선 마우스', 'price': 22000, 'stock': 15, 'created_at': '2026-11-12T02:18:28.800391+00:00'}]


## Delete: 상품 삭제 (관리자)

In [6]:
res = admin.table("products").delete().eq("name", "마우스 패드").execute()
print("삭제된 상품:", [p["name"] for p in res.data])

print("남은 상품:", [p["name"] for p in guest.table("products").select("name").order("id").execute().data])

삭제된 상품: ['마우스 패드']


남은 상품: ['무선 마우스', '기계식 키보드', '27인치 모니터']


## 예외: 일반 회원이 등록 / 수정 → RLS로 막힘

In [7]:
try:
    user1.table("products").insert({"name": "몰래 등록", "price": 1000, "stock": 1}).execute()
except Exception as e:
    print("등록 실패:", e)

res = user1.table("products").update({"price": 0}).eq("name", "무선 마우스").execute()
print("수정 시도 → 바뀐 행:", len(res.data))

등록 실패: {'message': 'new row violates row-level security policy for table "products"', 'code': '42501', 'hint': None, 'details': None}


수정 시도 → 바뀐 행: 0


## 예외: 제약조건 위반 (가격 음수) → CHECK로 막힘

In [8]:
try:
    admin.table("products").insert({"name": "이상한 상품", "price": -100, "stock": 1}).execute()
except Exception as e:
    print("등록 실패:", e)

등록 실패: {'message': 'new row for relation "products" violates check constraint "products_price_check"', 'code': '23514', 'hint': None, 'details': None}
